# Notebook C-fix — Metadata Encoder Fix and Targeted Rerun (Task B)

**Why this notebook exists.** Notebook A's GBM audit shows exact GPS carries a genuine
8.4-point accuracy lift for Task B (0.213 vs 0.129 majority). Notebook C's neural
metadata-only control found almost nothing (0.130), and got the `country` vs `climate`
ordering backwards — the signature of an encoder that cannot extract structure from a
234-way one-hot vector fed through a single 64-unit linear layer, not evidence the
information is absent.

**What changes here, precisely two things:**
1. `country_code` gets an `nn.Embedding` instead of one-hot + linear. Embeddings scale to
   high cardinality; a linear layer on a sparse one-hot does not.
2. The metadata branch gets more capacity (128 units, two layers instead of one).

**One extra diagnostic, added because it directly answers the open question.** Even with
a fixed encoder, joint training can still starve the metadata branch of gradient if the
image branch is easier to optimise — a documented failure mode in multimodal fusion
(modality dominance / greedy modality learning; Wang et al., CVPR 2020, and follow-ups —
verify current framing before citing). So alongside the capacity fix, one condition is
run with **modality dropout**: during training, the image branch is zeroed for a fraction
of batches, forcing the network to route gradient through the metadata branch some of the
time. Comparing plain-fixed-encoder vs fixed-encoder-plus-modality-dropout tells you
*which* explanation is true if synergy still doesn't show up:

| Outcome | Diagnosis |
|---|---|
| capacity fix alone recovers synergy | it was purely an encoder problem |
| capacity fix alone doesn't, dropout version does | modality dominance was suppressing a branch that could learn |
| neither recovers synergy | the fusion architecture genuinely can't extract what GBM found — worth a sentence in Limitations, and grounds to report the GBM ceiling directly instead |

**Scope.** 9 core conditions + 1 modality-dropout condition, Task B only, spatial protocol
only, 5 seeds — 50 runs, under an hour on GPU. Checkpointed like Notebook C.

In [ ]:
import json, time, glob, warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

AUD_DIR = Path('/kaggle/input/nba-audit-splits')      # Notebook A output
OLD_RUNS = Path('/kaggle/input/nbc-runs/runs.csv')     # original Notebook C output, for comparison
OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True, parents=True)
PRED_DIR = OUT / 'preds'; PRED_DIR.mkdir(exist_ok=True, parents=True)
CKPT_INPUTS = [
    # Path('/kaggle/input/nbcfix-part1'),
]

SEEDS = [0, 1, 2, 3, 4]
EPOCHS, PATIENCE, BATCH, LR, WD = 60, 8, 512, 1e-3, 1e-4
IMG_H, META_H, COUNTRY_EMB, FUSE_H, DROP = 256, 128, 32, 128, 0.3
TIME_BUDGET_H = 3.0
T0 = time.time()
elapsed_h = lambda: (time.time() - T0) / 3600
print('device:', DEVICE)

In [ ]:
# ── Load frozen artefacts from Notebook A ───────────────────────────────
df   = pd.read_parquet(AUD_DIR / 'corpus.parquet')
EMB  = np.load(AUD_DIR / 'resnet_emb.npy').astype(np.float32)
META = json.loads((AUD_DIR / 'splits.json').read_text())
SPL_B = {p: np.array(v) for p, v in META['splits']['spatial'].items()}  # Task B, spatial only
y = df.yB.values.astype(np.int64)
CLASSES = META['classesB']
print(f'{len(df):,} rows | Task B | spatial split '
      f"train={len(SPL_B['train']):,} val={len(SPL_B['val']):,} test={len(SPL_B['test']):,}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  METADATA BUILDER v2 — embeddings for country, larger numeric branch
# ════════════════════════════════════════════════════════════════════════
COUNTRY_VOCAB = sorted(df.country_code.astype(str).unique())
COUNTRY_IDX_MAP = {c: i for i, c in enumerate(COUNTRY_VOCAB)}
N_COUNTRY = len(COUNTRY_VOCAB)
CLIMATE_VOCAB = sorted(df.climate_zone.astype(str).unique())
HEMI_VOCAB = sorted(df.hemisphere.astype(str).unique())
print(f'country vocab: {N_COUNTRY} | climate: {len(CLIMATE_VOCAB)} | hemisphere: {len(HEMI_VOCAB)}')

def build_meta_v2(groups, permute_all, seed):
    """Returns (numeric_X [N,d], country_idx [N] int64 or None).
    'country' group -> embedding index array, everything else -> the numeric matrix.
    permute_all shuffles both together, identically, breaking the id<->metadata link
    for the permutation control (P0/P1)."""
    rng = np.random.RandomState(2000 + seed)
    num_parts, names = [], []
    country_idx = None

    if 'gps' in groups:
        lat, lon = df.lat.values, df.lon.values
        num_parts.append(np.stack([lat/90.0, lon/180.0,
                                   np.sin(np.radians(lat)), np.sin(np.radians(lon)),
                                   np.cos(np.radians(lon))], axis=1))
        names += ['lat_n', 'lon_n', 'lat_sin', 'lon_sin', 'lon_cos']
    if 'geo_derived' in groups:
        cl = pd.get_dummies(df.climate_zone.astype(str)).reindex(columns=CLIMATE_VOCAB, fill_value=0)
        he = pd.get_dummies(df.hemisphere.astype(str)).reindex(columns=HEMI_VOCAB, fill_value=0)
        num_parts.append(np.hstack([cl.values, he.values]).astype(np.float32))
        names += [f'climate={c}' for c in CLIMATE_VOCAB] + [f'hemi={h}' for h in HEMI_VOCAB]
    if 'file_meta' in groups:
        num_parts.append(df[['aspect', 'megapix', 'bytes_px']].values.astype(np.float32))
        names += ['aspect', 'megapix', 'bytes_px']
    if 'country' in groups:
        country_idx = df.country_code.astype(str).map(COUNTRY_IDX_MAP).values.astype(np.int64)

    numeric = np.hstack(num_parts).astype(np.float32) if num_parts else np.zeros((len(df), 0), np.float32)

    if permute_all:
        n = len(df)
        order = rng.permutation(n)          # one shared permutation -> record link fully destroyed
        numeric = numeric[order]
        if country_idx is not None:
            country_idx = country_idx[order]

    return numeric, country_idx, names

# self-test
n1, c1, nm1 = build_meta_v2(['gps', 'geo_derived'], False, 0)
n2, c2, nm2 = build_meta_v2(['country'], False, 0)
n3, c3, nm3 = build_meta_v2(['gps', 'geo_derived'], True, 0)
print('gps+geo_derived numeric:', n1.shape, nm1)
print('country embedding idx:', c2.shape, c2.min(), c2.max())
print('permuted differs from original:', not np.allclose(n1, n3))

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  MODEL — separate embedding branch for country, larger numeric branch,
#  optional modality dropout on the image branch during training
# ════════════════════════════════════════════════════════════════════════
class FusionV2(nn.Module):
    def __init__(self, img_dim, num_dim, use_country, n_classes,
                 img_h=IMG_H, meta_h=META_H, country_emb=COUNTRY_EMB, fuse_h=FUSE_H, p=DROP):
        super().__init__()
        self.img_dim, self.num_dim, self.use_country = img_dim, num_dim, use_country
        fuse_in = 0
        if img_dim:
            self.img = nn.Sequential(nn.Linear(img_dim, img_h), nn.BatchNorm1d(img_h),
                                     nn.ReLU(), nn.Dropout(p))
            fuse_in += img_h
        if num_dim:
            self.num = nn.Sequential(nn.Linear(num_dim, meta_h), nn.BatchNorm1d(meta_h), nn.ReLU(), nn.Dropout(p),
                                     nn.Linear(meta_h, meta_h), nn.BatchNorm1d(meta_h), nn.ReLU(), nn.Dropout(p))
            fuse_in += meta_h
        if use_country:
            self.emb = nn.Embedding(N_COUNTRY, country_emb)
            self.emb_proj = nn.Sequential(nn.Linear(country_emb, meta_h), nn.BatchNorm1d(meta_h),
                                          nn.ReLU(), nn.Dropout(p))
            fuse_in += meta_h
        self.head = nn.Sequential(nn.Linear(fuse_in, fuse_h), nn.ReLU(), nn.Dropout(p),
                                  nn.Linear(fuse_h, n_classes))

    def forward(self, xi, xn, xc, drop_image_mask=None):
        zs = []
        if self.img_dim:
            zi = self.img(xi)
            if drop_image_mask is not None:
                zi = zi * (~drop_image_mask).float().unsqueeze(1)   # zero out for dropped rows
            zs.append(zi)
        if self.num_dim:
            zs.append(self.num(xn))
        if self.use_country:
            zs.append(self.emb_proj(self.emb(xc)))
        return self.head(torch.cat(zs, dim=1))

def std(X, tr):
    if X.shape[1] == 0: return X
    mu, sd = X[tr].mean(0), X[tr].std(0); sd[sd == 0] = 1.
    return ((X - mu)/sd).astype(np.float32)

In [ ]:
# ── Train / eval, with optional modality dropout ────────────────────────
def run_one(use_img, groups, permute_all, modality_dropout_p, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    tr, va, te = SPL_B['train'], SPL_B['val'], SPL_B['test']

    Xn, Xc, _ = build_meta_v2(groups, permute_all, seed)
    Xn = std(Xn, tr)
    Xi = std(EMB, tr) if use_img else np.zeros((len(df), 0), np.float32)
    use_country = Xc is not None
    Xc_t = torch.from_numpy(Xc) if use_country else torch.zeros(len(df), dtype=torch.long)

    def ld(ix, sh):
        return DataLoader(TensorDataset(torch.from_numpy(Xi[ix]), torch.from_numpy(Xn[ix]),
                          Xc_t[ix], torch.from_numpy(y[ix])),
                          batch_size=BATCH, shuffle=sh, drop_last=sh and len(ix) > BATCH)
    dtr, dva, dte = ld(tr, True), ld(va, False), ld(te, False)

    mdl = FusionV2(Xi.shape[1], Xn.shape[1], use_country, len(CLASSES)).to(DEVICE)
    cnt = np.bincount(y[tr], minlength=len(CLASSES)).astype(np.float32)
    crit = nn.CrossEntropyLoss(weight=torch.tensor(cnt.sum()/(len(CLASSES)*np.maximum(cnt,1)), device=DEVICE))
    opt = torch.optim.AdamW(mdl.parameters(), lr=LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

    def ev(dl):
        mdl.eval(); P, Y = [], []
        with torch.no_grad():
            for xi, xn, xc, yy in dl:
                P.append(mdl(xi.to(DEVICE), xn.to(DEVICE), xc.to(DEVICE)).argmax(1).cpu().numpy())
                Y.append(yy.numpy())
        return np.concatenate(P), np.concatenate(Y)

    best, bad, state, ep = -1, 0, None, 0
    do_drop = use_img and (Xn.shape[1] or use_country) and modality_dropout_p > 0
    for ep in range(EPOCHS):
        mdl.train()
        for xi, xn, xc, yy in dtr:
            mask = (torch.rand(xi.shape[0]) < modality_dropout_p) if do_drop else None
            opt.zero_grad()
            out = mdl(xi.to(DEVICE), xn.to(DEVICE), xc.to(DEVICE),
                      drop_image_mask=mask.to(DEVICE) if mask is not None else None)
            crit(out, yy.to(DEVICE)).backward(); opt.step()
        sch.step()
        pv, yv = ev(dva); f = f1_score(yv, pv, average='macro')
        if f > best + 1e-5:
            best, bad = f, 0; state = {a: b.detach().clone() for a, b in mdl.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    mdl.load_state_dict(state)
    pt, yt = ev(dte)
    return dict(test_acc=accuracy_score(yt, pt), test_macro_f1=f1_score(yt, pt, average='macro'),
                test_bal_acc=balanced_accuracy_score(yt, pt), val_macro_f1=best, epochs=ep+1,
                meta_dim=int(Xn.shape[1]) + (COUNTRY_EMB if use_country else 0),
                n_params=sum(p.numel() for p in mdl.parameters())), pt.astype(np.int16), yt.astype(np.int16)

# smoke test
t0 = time.time()
m, _, _ = run_one(True, ['gps', 'geo_derived'], False, 0.0, 0)
print(f'smoke test: acc={m["test_acc"]:.4f} f1={m["test_macro_f1"]:.4f} ({time.time()-t0:.0f}s, {m["epochs"]} epochs)')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  JOB REGISTRY — 9 core conditions + 1 modality-dropout diagnostic
# ════════════════════════════════════════════════════════════════════════
# (run_id_base, label, use_img, groups, permute_all, modality_dropout_p)
JOBS_DEF = [
    ('C0',  'Image only',                          True,  [],                      False, 0.0),
    ('C4',  'Image + GPS exact (fixed encoder)',    True,  ['gps'],                 False, 0.0),
    ('C5',  'Image + GPS + climate (fixed)',        True,  ['gps', 'geo_derived'],  False, 0.0),
    ('C6',  'Image + all metadata (fixed)',         True,  ['gps', 'geo_derived', 'file_meta', 'country'], False, 0.0),
    ('M0',  'Metadata only: GPS (fixed)',           False, ['gps'],                 False, 0.0),
    ('M1',  'Metadata only: primary (fixed)',       False, ['gps', 'geo_derived'],  False, 0.0),
    ('M3',  'Metadata only: country (fixed)',       False, ['country'],             False, 0.0),
    ('P0',  'Image + PERMUTED GPS (fixed)',         True,  ['gps'],                 True,  0.0),
    ('P1',  'Image + PERMUTED primary (fixed)',     True,  ['gps', 'geo_derived'],  True,  0.0),
    # bonus diagnostic: does forcing metadata use recover synergy?
    ('C5d', 'Image + GPS + climate, modality-dropout p=0.3', True, ['gps', 'geo_derived'], False, 0.3),
]
JOBS = [dict(run_id=f'{cid}|s{s}', **{'condition': cid, 'label': lab, 'use_image': ui,
             'groups': g, 'permute_all': pa, 'modality_dropout_p': mp, 'seed': s})
        for cid, lab, ui, g, pa, mp in JOBS_DEF for s in SEEDS]
print(f'{len(JOBS)} runs planned ({len(JOBS_DEF)} conditions x {len(SEEDS)} seeds)')

In [ ]:
# ── Run with checkpoint / resume ─────────────────────────────────────────
def load_done():
    frames, seen = [], set()
    for d in [OUT] + [Path(p) for p in CKPT_INPUTS]:
        f = Path(d) / 'runs_fix.csv'
        if f.exists():
            r = pd.read_csv(f); frames.append(r); seen |= set(r.run_id)
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), seen

done_df, done = load_done()
print(f'{len(done)} runs already complete, {len(JOBS)-len(done)} remaining')

rows = []
for n, j in enumerate(JOBS):
    if j['run_id'] in done:
        continue
    if elapsed_h() > TIME_BUDGET_H:
        print('TIME BUDGET reached — save a version, publish, add to CKPT_INPUTS, re-run.'); break
    m, pt, yt = run_one(j['use_image'], j['groups'], j['permute_all'], j['modality_dropout_p'], j['seed'])
    np.savez_compressed(PRED_DIR / f"{j['run_id'].replace('|','_')}.npz", pred=pt, label=yt)
    rows.append({k: v for k, v in j.items() if k not in ('groups',)} |
                {'groups': '+'.join(j['groups']) or 'none'} | m)
    if len(rows) % 5 == 0 or n == len(JOBS) - 1:
        pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True).to_csv(OUT/'runs_fix.csv', index=False)
        print(f"[{n+1}/{len(JOBS)}] {j['run_id']:10s} acc={m['test_acc']:.4f} f1={m['test_macro_f1']:.4f} "
              f"| {elapsed_h():.2f}h", flush=True)

runs_fix = pd.concat([done_df, pd.DataFrame(rows)], ignore_index=True)
runs_fix.to_csv(OUT / 'runs_fix.csv', index=False)
print(f'\n{len(runs_fix)}/{len(JOBS)} runs complete')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  IMMEDIATE VERDICT — old (one-hot) vs new (embedding) vs GBM ceiling
# ════════════════════════════════════════════════════════════════════════
def summ(r, cid):
    g = r[r.condition == cid]
    return g.test_acc.mean(), g.test_acc.std(), g.test_macro_f1.mean()

print(f"{'condition':6s} {'new acc':>10s} {'new f1':>10s}")
for cid in ['C0','C4','C5','C6','M0','M1','M3','P0','P1','C5d']:
    if (runs_fix.condition == cid).any():
        a, s, f = summ(runs_fix, cid)
        print(f'{cid:6s} {a:10.4f} {f:10.4f}  (+/-{s:.4f})')

if OLD_RUNS.exists():
    old = pd.read_csv(OLD_RUNS)
    old_b = old[old.task == 'B']
    print('\n--- old (one-hot) vs new (embedding) metadata-only accuracy ---')
    for cid, gbm_lift in [('M0', 0.0844), ('M1', None), ('M3', 0.0336)]:
        oa = old_b[old_b.condition == cid].test_acc.mean()
        na = summ(runs_fix, cid)[0] if (runs_fix.condition == cid).any() else np.nan
        print(f'{cid}: old={oa:.4f}  new={na:.4f}  delta={na-oa:+.4f}'
              + (f'   (GBM ceiling lift was +{gbm_lift:.4f})' if gbm_lift else ''))

    print('\n--- synergy recomputed with the fixed encoder ---')
    ai = summ(runs_fix, 'C0')[0]
    for fus, ctrl in [('C4','M0'), ('C5','M1'), ('C6','M3')]:
        if (runs_fix.condition == fus).any() and (runs_fix.condition == ctrl).any():
            af = summ(runs_fix, fus)[0]; am = summ(runs_fix, ctrl)[0]
            S = af - max(ai, am); L = am/af if af else np.nan
            print(f'{fus} (ctrl {ctrl}): Acc(image)={ai:.4f} Acc(meta)={am:.4f} Acc(fusion)={af:.4f} '
                  f'-> S={S:+.4f}  L={L:.3f}  {"GENUINE SYNERGY" if S>0.01 else "still no gain" if S>-0.01 else "harms"}')

    if (runs_fix.condition == 'C5d').any():
        a5, _, f5 = summ(runs_fix, 'C5')
        a5d, _, f5d = summ(runs_fix, 'C5d')
        print(f'\n--- modality dropout diagnostic ---')
        print(f'C5  (no dropout):     acc={a5:.4f}  f1={f5:.4f}')
        print(f'C5d (dropout p=0.3):  acc={a5d:.4f}  f1={f5d:.4f}   delta={a5d-a5:+.4f}')
        print('If C5d clearly beats C5, gradient starvation was suppressing a branch that could learn.')
        print('If C5d ~= C5, capacity was the whole story (or the ceiling really is this low).')
else:
    print('\nAttach the original nbC output (runs.csv) as a Kaggle Dataset and set OLD_RUNS '
          'to compare against the pre-fix numbers directly.')

---
## Reading the verdict

- **If `M0`/`M1`/`M3` new accuracy moved substantially toward the GBM ceiling** (0.213 /
  ~0.20 / 0.162), the encoder fix worked — the metadata branch can now see the signal GBM
  already proved exists. Check whether `S` for `C4`/`C5`/`C6` turned positive.
- **If `M0`/`M1`/`M3` are still near chance despite the embedding**, something structural
  remains wrong — check `n_params` and `meta_dim` in `runs_fix.csv` to confirm the
  embedding branch is actually being used, and consider a plain GBM feature-importance
  check to make sure Notebook A's audit and this rerun are seeing the same underlying columns.
- **The `C5` vs `C5d` comparison settles the modality-dominance question directly**, independent
  of the capacity question.

**Next step either way:** don't touch the dose-response grid until this cell's synergy
numbers are decided. If synergy recovers, rerun the quality grid (Notebook C's `QUALITY_GRID`
section) with this fixed encoder before drafting Section 4.5. If it doesn't, Section 4.5
gets dropped or replaced with the GBM-ceiling-vs-neural-floor comparison instead — send me
this notebook's printed verdict and we'll decide the Results section's shape together.